# OOS Backtest + Coverage Grid (SPY): Same-day vs Out-of-sample (1-step shift)


**Updated:** 2026-03-16  
**Goal:**
1) Compare **same-day** vs **out-of-sample (OOS, 1-step shift)** VaR backtests  
2) Run a small **coverage grid** over $(\alpha,\ \mathrm{window})$ and summarize results in a compact table  
3) Save outputs for reporting (CSV + optional plot)

## Conventions
- Log return (used as PnL): $$ r_t = \log(P_t) - \log(P_{t-1}) $$
- Loss: $$ L_t = -r_t $$
- VaR is reported as **positive loss magnitude**

## Backtesting timing: same-day vs OOS

### Same-day (contemporaneous)
Rolling historical VaR at time $t$ uses a window that **includes $t$**:

$$ \mathrm{VaR}_{\alpha,t} = q_\alpha(L_{t-w+1}, \dots, L_t) $$

Violation:

$$ I_t^{\mathrm{same}} = \mathbf{1}_{\{L_t > \mathrm{VaR}_{\alpha,t}\}} $$

### Out-of-sample (OOS, 1-step shift)
Evaluate day $t$ using a VaR estimated from information available up to $t-1$:

$$ I_t^{\mathrm{oos}} = \mathbf{1}_{\{L_t > \mathrm{VaR}_{\alpha,t-1}\}} $$

In code, if `rolling_historical_var(...)` returns $\mathrm{VaR}_{\alpha,t}$, then:
- OOS is implemented by `VaR.shift(1)`.

## 1) Import & Data: SPY price → log returns → loss

We load SPY daily prices, convert them to log returns (PnL), and then define loss by the project convention:

- $$ r_t = \log(P_t) - \log(P_{t-1}) $$
- $$ L_t = -r_t $$

In [6]:
import numpy as np
import pandas as pd

from riskmetrics.var import rolling_historical_var
from riskmetrics.backtest import backtest_report, backtest_report_oos

df = pd.read_csv("../data/price_SPY.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").set_index("date")

price = df["price"].astype(float)

pnl = np.log(price).diff().dropna()   # log returns as PnL
loss = -pnl                           # loss convention

print("n_prices:", len(price), "n_returns:", len(pnl))
loss.head()

n_prices: 1256 n_returns: 1255


date
2021-03-09   -0.014177
2021-03-10   -0.006205
2021-03-11   -0.010088
2021-03-12   -0.001346
2021-03-15   -0.005946
Name: price, dtype: float64

## 2) Single-setting backtest (one $(\alpha, window)$)

We compute rolling historical VaR and compare:
- same-day: $L_t$ vs $\mathrm{VaR}_{\alpha,t}$
- OOS: $L_t$ vs $\mathrm{VaR}_{\alpha,t-1}$

This is a quick check before running the full grid.

In [7]:
# knobs
alpha = 0.99
window = 250

rvar = rolling_historical_var(pnl, window=window, alpha=alpha)

rep_same = backtest_report(loss, rvar, alpha=alpha)
rep_oos  = backtest_report_oos(loss, rvar, alpha=alpha)

print("=== SPY SAME-DAY ===")
print(rep_same)
print("\n=== SPY OOS (shift1) ===")
print(rep_oos)

=== SPY SAME-DAY ===
{'n': 1006, 'x': 19, 'expected_rate': 0.010000000000000009, 'observed_rate': 0.018886679920477135, 'lr_pof': 6.363619565091511, 'p_value': 0.011648368066473513}

=== SPY OOS (shift1) ===
{'n': 1005, 'x': 19, 'expected_rate': 0.010000000000000009, 'observed_rate': 0.01890547263681592, 'lr_pof': 6.38167266408999, 'p_value': 0.01153047141963337}


## 3) Coverage grid (multiple $(\alpha, window)$)

We run a small grid:
- $$ \alpha \in \{0.975,\ 0.99\} $$
- $$ window \in \{125,\ 250,\ 500\} $$

For each setting we record:
- $n$ (valid days), $x$ (violations)
- expected rate $(1-\alpha)$ vs observed rate $(x/n)$
- Kupiec POF LR statistic and p-value

In [8]:
alphas = [0.975, 0.99]
windows = [125, 250, 500]

rows = []
for a in alphas:
    for w in windows:
        rvar = rolling_historical_var(pnl, window=w, alpha=a)

        rep_same = backtest_report(loss, rvar, alpha=a)
        rep_same.update({"alpha": a, "window": w, "mode": "same_day"})

        rep_oos = backtest_report_oos(loss, rvar, alpha=a)
        rep_oos.update({"alpha": a, "window": w, "mode": "oos_shift1"})

        rows.append(rep_same)
        rows.append(rep_oos)

out = (
    pd.DataFrame(rows)
    .sort_values(["alpha", "window", "mode"])
    .reset_index(drop=True)
)

out = out[["alpha", "window", "mode", "n", "x", "expected_rate", "observed_rate", "lr_pof", "p_value"]]
out

,alpha,window,mode,n,x,expected_rate,observed_rate,lr_pof,p_value
0,0.975,125,oos_shift1,1130,47,0.025,0.041593,10.672010,0.001088
1,0.975,125,same_day,1131,46,0.025,0.040672,9.609478,0.001936
2,0.975,250,oos_shift1,1005,34,0.025,0.033831,2.900434,0.088556
3,0.975,250,same_day,1006,33,0.025,0.032803,2.291872,0.130053
4,0.975,500,oos_shift1,755,19,0.025,0.025166,0.000847,0.976779
5,0.975,500,same_day,756,19,0.025,0.025132,0.000542,0.981431
6,0.990,125,oos_shift1,1130,25,0.010,0.022124,12.472119,0.000413
7,0.990,125,same_day,1131,24,0.010,0.021220,10.877962,0.000973
8,0.990,250,oos_shift1,1005,19,0.010,0.018905,6.381673,0.011530
9,0.990,250,same_day,1006,19,0.010,0.018887,6.363620,0.011648


## 4) Summary

### Window length
- Short windows often have noisier tail estimates, so the observed violation rate can exceed the expected rate (under-coverage).
- Longer windows stabilize empirical quantiles and generally improve coverage (observed rate moves closer to expected).

### Confidence level
- $\alpha = 0.99$ is harder because it targets the extreme tail.
- Extreme-tail estimation is sensitive to heavy tails and volatility clustering, which are common in real market returns.

### Same-day vs OOS
- OOS uses one fewer effective observation due to the 1-step shift ($\mathrm{VaR}_{\alpha,t-1}$ aligned to day $t$).
- Differences between same-day and OOS are often small for large windows. (The dominant effect is typically the $(\alpha, \mathrm{window})$ choice.)
- OOS is preferred as a leakage-safe evaluation: compare $L_t$ against $\mathrm{VaR}_{\alpha,t-1}$.

### Practical takeaway
- A compact grid table makes it easy to select $(\alpha, \mathrm{window})$ settings that produce violation rates closer to the expected $(1-\alpha)$.
- For SPY, short windows (e.g., 125) tend to under-cover and are often rejected by Kupiec POF, while longer windows (e.g., 500) can yield more stable coverage.

---